In [1]:
#1. Genarate dataset
#2.Train the classifier
#3. Detect face and name it is already in dataset

# Genarate Dataset


In [2]:
import cv2
import os

In [11]:


def generate_dataset():

    # Load Haar Cascade
    face_classifier = cv2.CascadeClassifier(
        "haarcascade_frontalface_default.xml"
    )

    # Face crop function
    def face_cropped(img):
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

        faces = face_classifier.detectMultiScale(
            gray,
            scaleFactor=1.3,
            minNeighbors=5
        )

        if len(faces) == 0:
            return None

        for (x, y, w, h) in faces:
            cropped_face = img[y:y+h, x:x+w]

        return cropped_face

    # Create data folder if it doesn't exist
    if not os.path.exists("data"):
        os.makedirs("data")

    # -------- AUTO GENERATE NEXT PERSON ID --------
    existing_files = os.listdir("data")

    ids = []

    for file in existing_files:
        if file.startswith("user."):
            try:
                parts = file.split(".")
                ids.append(int(parts[1]))
            except:
                pass

    person_id = max(ids) + 1 if ids else 1

    print(f"Starting capture for Person ID: {person_id}")

    # ----------------------------------------------

    cap = cv2.VideoCapture(0)

    img_id = 0

    while True:

        ret, frame = cap.read()

        if not ret:
            break

        cropped = face_cropped(frame)

        if cropped is not None:

            img_id += 1

            face = cv2.resize(cropped, (200, 200))
            face = cv2.cvtColor(face, cv2.COLOR_BGR2GRAY)

            file_name_path = f"data/user.{person_id}.{img_id}.jpg"

            cv2.imwrite(file_name_path, face)

            cv2.putText(
                face,
                str(img_id),
                (50, 50),
                cv2.FONT_HERSHEY_COMPLEX,
                1,
                (255, 255, 255),
                2
            )

            cv2.imshow("Cropped Face", face)

        # Stop at Enter key OR 200 images
        if cv2.waitKey(1) == 13 or img_id == 200:
            break

    cap.release()
    cv2.destroyAllWindows()

    print(f"Dataset collection completed for Person ID: {person_id}")

# Run function
generate_dataset()

Starting capture for Person ID: 4
Dataset collection completed for Person ID: 4


# Train the classifier

In [2]:
import os
import cv2
from PIL import Image #pip install pillow
import numpy as np    # pip install numpy

In [13]:
def train_classifier(data_dir):
    path = [os.path.join(data_dir, f) for f in os.listdir(data_dir)]
    
    faces = []
    ids = []
    
    for image in path:
        img = Image.open(image).convert('L')
        imageNp = np.array(img, 'uint8')
        id = int(os.path.split(image)[1].split(".")[1])
        
        faces.append(imageNp)
        ids.append(id)
        
    ids = np.array(ids)
    
    # Train and save classifier
    clf = cv2.face.LBPHFaceRecognizer_create()
    clf.train(faces,ids)
    clf.write("classifier.xml")
train_classifier("data")

In [3]:
def draw_boundary(img, classifier, scaleFactor, minNeighbors, color, text, clf):
    gray_img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    features = classifier.detectMultiScale(gray_img, scaleFactor, minNeighbors)
    
    for (x,y,w,h) in features:
        cv2.rectangle(img, (x,y), (x+w,y+h), color, 2 )
        
        id, pred = clf.predict(gray_img[y:y+h,x:x+w])
        confidence = int(100*(1-pred/300))
        
        if confidence>70:
            if id==1:
                cv2.putText(img, "Anika", (x,y-5), cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 1, cv2.LINE_AA)
            if id==4:
                cv2.putText(img, "Akib", (x,y-5), cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 1, cv2.LINE_AA)
            if id==3:
                cv2.putText(img, "Samad", (x,y-5), cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 1, cv2.LINE_AA)
        else:
            cv2.putText(img, "UNKNOWN", (x,y-5), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,0,255), 1, cv2.LINE_AA)
    
    return img

# loading classifier
faceCascade = cv2.CascadeClassifier("haarcascade_frontalface_default.xml")

clf = cv2.face.LBPHFaceRecognizer_create()
clf.read("classifier.xml")

video_capture = cv2.VideoCapture(0)

while True:
    ret, img = video_capture.read()
    img = draw_boundary(img, faceCascade, 1.3, 6, (255,255,255), "Face", clf)
    cv2.imshow("face Detection", img)
    
    if cv2.waitKey(1)==13:
        break
video_capture.release()
cv2.destroyAllWindows()